In [ ]:
import math

# 1. Función a integrar en [-1, 1]
def carga_ala(x):
    return math.sqrt(1 - x**2) * math.exp(-0.5 * x**2)

# 2. Polinomio de Legendre P_n(x) por recurrencia de tres términos:
#    (k+1) P_{k+1} = (2k+1) x P_k - k P_{k-1},  con P_0 = 1, P_1 = x
#    Devuelve (P_n(x), P_{n-1}(x))
def legendre(n, x):
    p_prev, p_curr = 1.0, x
    for k in range(1, n):
        p_next = ((2 * k + 1) * x * p_curr - k * p_prev) / (k + 1)
        p_prev, p_curr = p_curr, p_next
    return p_curr, p_prev

# 3. Nodos y pesos de Gauss-Legendre
#    Nodos:  x_i = raíces de P_n(x)  (Newton-Raphson)
#    Pesos:  w_i = 2 / [ (1 - x_i^2) * (P_n'(x_i))^2 ]
#    Derivada: P_n'(x) = n (x P_n - P_{n-1}) / (x^2 - 1)
def nodos_pesos(n, tol=1e-15, max_iter=100):
    nodos, pesos = [], []
    for i in range(n):
        # Aproximación inicial de la i-ésima raíz
        x = math.cos(math.pi * (i + 0.75) / (n + 0.5))
        for _ in range(max_iter):
            pn, pn1 = legendre(n, x)
            dpn = n * (x * pn - pn1) / (x * x - 1)
            dx = pn / dpn
            x -= dx
            if abs(dx) < tol:
                break
        pn, pn1 = legendre(n, x)
        dpn = n * (x * pn - pn1) / (x * x - 1)
        nodos.append(x)
        pesos.append(2.0 / ((1 - x * x) * dpn**2))
    return nodos, pesos

# 4. Cuadratura: I ≈ sum_i w_i f(x_i)
def gauss_legendre(f, n):
    x, w = nodos_pesos(n)
    return sum(wi * f(xi) for xi, wi in zip(x, w))

# 5. Valor de referencia (sin scipy)
#    Con x = sen(t) el integrando se vuelve suave:
#    I = ∫_{-π/2}^{π/2} cos²(t) · exp(-0.5 sen²(t)) dt
#    y se integra con Simpson compuesto de muchos subintervalos.
def referencia(N=20000):
    g = lambda t: math.cos(t)**2 * math.exp(-0.5 * math.sin(t)**2)
    a, b = -math.pi / 2, math.pi / 2
    h = (b - a) / N
    s = g(a) + g(b)
    for k in range(1, N):
        s += (4 if k % 2 else 2) * g(a + k * h)
    return s * h / 3

valor_exacto = referencia()

print(f"Sustentación total exacta: {valor_exacto:.10f}\n")
print(f"{'Nodos (n)':<10} | {'Resultado Gauss':<16} | {'Error Absoluto':<15}")
print("-" * 48)

for n in [2, 4, 8, 16, 32]:
    aprox = gauss_legendre(carga_ala, n)
    error = abs(valor_exacto - aprox)
    print(f"{n:<10} | {aprox:<16.10f} | {error:<15.2e}")